In [ ]:
# Setup: clone or pull logseer repo and import core library
import os, sys
if not os.path.exists('logseer'):
    !git clone https://github.com/masahiko-shibata/logseer.git
else:
    !cd logseer && git pull
sys.path.insert(0, 'logseer')
from logseer import *
import keras

In [ ]:
# Load log data from Google Drive
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

!rm -rf logs data.zip 2>/dev/null
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/data/data.zip', 'data.zip')
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
print(os.listdir('./'))

In [ ]:
# Configuration
import numpy as np
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.models import load_model
from keras import optimizers
import xgboost as xgb
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import binomtest

BASE_DIR            = './'
DATA_DIR            = BASE_DIR + 'data'
MAX_NB_WORDS        = 24000
MAX_SEQUENCE_LENGTH = 26000
EMBEDDING_DIM       = 32
SUCCESS_LOG_RATIO   = 99
VALIDATION_SPLIT    = 0.1
VALIDATE_ON_TEST_DATA = False
EPOCHS              = 60
BATCH_SIZE          = 32
MODEL_SAVE_PATH     = 'logseer.keras'
TOKENIZER_PATH      = 'tokenizer.pickle'
EMBEDDING_LAYER     = 'vanilla'
MODEL_NAME          = 'LogCNNLite'
REPETITION          = 100
ERROR_WEIGHT        = 2
LEARNING_RATE       = 0.0003
MAX_LOSS            = 0.7
PATIENCE            = 15
RESTORE_BEST_WEIGHTS = False
START_FROM_EPOCH    = 20
MONITOR             = 'val_recall'
MODE                = 'max'
RETRAIN             = False
test_NN             = True
test_xgb            = True
test_svm            = False
test_rf             = False

In [ ]:
# Training loop
ld = Loader()
tester = Tester()

for i in range(REPETITION):
    print()
    print('Repetition %s' % str(i + 1))
    texts, labels, test_texts, test_labels = ld.getdata(DATA_DIR)
    print('Found %s texts.' % len(texts))
    print('Found %s test texts.' % len(test_texts))

    if VALIDATE_ON_TEST_DATA:
        train_texts  = np.array(texts)
        train_labels = np.array(labels)
        val_texts    = np.array(test_texts)
        val_labels   = np.array(test_labels)
    else:
        nb_val_samples = int(VALIDATION_SPLIT * len(texts))
        train_texts  = np.array(texts[:-nb_val_samples])
        train_labels = np.array(labels[:-nb_val_samples])
        val_texts    = np.array(texts[-nb_val_samples:])
        val_labels   = np.array(labels[-nb_val_samples:])

    if RETRAIN:
        with open(TOKENIZER_PATH, 'rb') as handle:
            tokenizer = pickle.load(handle)
    else:
        tokenizer = Tokenizer(num_words=MAX_NB_WORDS, filters='', lower=False)
        tokenizer.fit_on_texts(train_texts)
        with open(TOKENIZER_PATH, 'wb') as handle:
            pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

    word_index = tokenizer.word_index
    print('Found %s unique tokens in the tokenizer.' % len(word_index))

    if test_NN:
        train_text_seqs = tokenizer.texts_to_sequences(train_texts)
        val_text_seqs   = tokenizer.texts_to_sequences(val_texts)
        test_text_seqs  = tokenizer.texts_to_sequences(test_texts)
        train_data = pad_sequences(train_text_seqs, maxlen=MAX_SEQUENCE_LENGTH)
        val_data   = pad_sequences(val_text_seqs,   maxlen=MAX_SEQUENCE_LENGTH)
        test_data  = pad_sequences(test_text_seqs,  maxlen=MAX_SEQUENCE_LENGTH)
        train_data = np.array(train_data, dtype=np.int32)
        val_data   = np.array(val_data,   dtype=np.int32)
        train_labels = np.array(train_labels, dtype=np.int32)
        val_labels   = np.array(val_labels,   dtype=np.int32)

        max_seq_len = max(len(s) for s in train_text_seqs + val_text_seqs)
        print('Longest data seq in train/val %s tokens' % max_seq_len)
        print('Shape of train data:', train_data.shape)

        nb_val_error = np.count_nonzero(val_labels)
        if nb_val_error < 1:
            print('Oops. Validation contains no error. Skipping this repetition.')
            continue

        print('Number of validation data sets: %s' % len(val_labels))
        print('Number of errors in validation data: %s' % nb_val_error)

        embedding_layer = getEmbeddingLayer(EMBEDDING_LAYER, MAX_NB_WORDS, EMBEDDING_DIM, MAX_SEQUENCE_LENGTH, word_index=word_index)
        optimizer = optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=0.9, beta_2=0.999)

        if RETRAIN:
            model = load_model(MODEL_SAVE_PATH)
        else:
            model = getModel(MODEL_NAME, embedding_layer, MAX_SEQUENCE_LENGTH, EMBEDDING_DIM)

        model.compile(loss='binary_crossentropy',
                      optimizer=optimizer,
                      metrics=[keras.metrics.Precision(name='precision'),
                               keras.metrics.Recall(name='recall')])

        checkpointer = MultiMetricCheckpoint(filepath=MODEL_SAVE_PATH, max_loss=MAX_LOSS)

        history = model.fit(train_data, train_labels,
                            validation_data=(val_data, val_labels),
                            epochs=EPOCHS,
                            verbose=1,
                            batch_size=BATCH_SIZE,
                            callbacks=[checkpointer]).history

        model = load_model(MODEL_SAVE_PATH)
        tester.testModel(model, test_data, test_labels, threshold=0.5)

    # Sklearn models — same training split as NN
    train_data_mat = tokenizer.texts_to_matrix(train_texts, mode='tfidf')
    test_data_mat  = tokenizer.texts_to_matrix(test_texts,  mode='tfidf')

    if test_xgb:
        xgbModel = xgb.XGBClassifier(scale_pos_weight=ERROR_WEIGHT)
        xgbModel.name = 'xgbModel'
        xgbModel.fit(train_data_mat, train_labels)
        pickle.dump(xgbModel, open('xgboost.pkl', 'wb'))
        tester.testModel(xgbModel, test_data_mat, test_labels)

    if test_svm:
        svmModel = svm.SVC(probability=True)
        svmModel.name = 'svmModel'
        svmModel.fit(train_data_mat, train_labels)
        pickle.dump(svmModel, open('svmModel.pkl', 'wb'))
        tester.testModel(svmModel, test_data_mat, test_labels)

    if test_rf:
        rfModel = RandomForestClassifier(n_jobs=2, random_state=0, n_estimators=50)
        rfModel.name = 'rfModel'
        rfModel.fit(train_data_mat, train_labels)
        pickle.dump(rfModel, open('rfModel.pkl', 'wb'))
        tester.testModel(rfModel, test_data_mat, test_labels)

    if i % 10 == 9:
        tester.total(heatmap=False)

    # Ensemble overlap: CNN vs XGB
    # Note: ensemble comparison is CNN vs XGB only. Enable test_xgb=True for meaningful ensemble metrics.
    nn_row  = next((r for r in tester.stored if r[0] not in ('xgbModel', 'svmModel', 'rfModel')), None)
    xgb_row = next((r for r in tester.stored if r[0] == 'xgbModel'), None)

    if nn_row and xgb_row:
        y    = np.array(nn_row[1])
        cnn  = np.array(nn_row[2])
        xgb_ = np.array(xgb_row[2])
        errors    = y == 1
        cnn_tp    = np.sum(errors & (cnn  == 1))
        xgb_tp    = np.sum(errors & (xgb_ == 1))
        both_tp   = np.sum(errors & (cnn  == 1) & (xgb_ == 1))
        either_tp = np.sum(errors & ((cnn == 1) | (xgb_ == 1)))
        either_fp = np.sum(~errors & ((cnn == 1) | (xgb_ == 1)))
        total_errors = np.sum(errors)
        ens_p  = either_tp / (either_tp + either_fp) if (either_tp + either_fp) > 0 else 0.0
        ens_r  = either_tp / total_errors if total_errors > 0 else 0.0
        ens_f1 = 2 * ens_p * ens_r / (ens_p + ens_r) if (ens_p + ens_r) > 0 else 0.0
        print()
        print('### Ensemble (CNN | XGB) ###')
        print()
        print(f'  Total errors   : {total_errors}')
        print(f'  CNN TP         : {cnn_tp}  (recall {cnn_tp/total_errors:.3f})')
        print(f'  XGB TP         : {xgb_tp}  (recall {xgb_tp/total_errors:.3f})')
        print(f'  Overlap (both) : {both_tp}')
        print(f'  CNN-only TP    : {cnn_tp - both_tp}')
        print(f'  XGB-only TP    : {xgb_tp - both_tp}')
        print(f'  Union TP       : {either_tp}')
        print(f'  Union FP       : {either_fp}')
        print(f'  Ensemble       : precision {ens_p:.3f}  recall {ens_r:.3f}  F1 {ens_f1:.3f}')

tester.total(heatmap=True)

print()
print('### Error Detection ###')
for row in tester.stored:
    name = row[0]
    y    = row[1]
    pred = row[2]
    n_errors        = sum(1 for label in y if label == 1)
    tp              = sum(1 for true, p in zip(y, pred) if true == 1 and p == 1)
    n_positive_preds = sum(1 for p in pred if p == 1)
    n_total         = len(pred)
    p_random = n_positive_preds / n_total
    result = binomtest(tp, n_errors, p_random, alternative='greater')
    print(f'{name}: TP={tp}/{n_errors}, p-value={result.pvalue:.6f}',
          '*** significant' if result.pvalue < 0.05 else '')

In [ ]:
# Save model and tokenizer to Google Drive
from google.colab import auth
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build('drive', 'v3')

def save_file_to_drive(name, path):
    file_metadata = {'name': name, 'mimeType': 'application/octet-stream'}
    media = MediaFileUpload(path, mimetype='application/octet-stream', resumable=True)
    created = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    print('File ID: {}'.format(created.get('id')))
    return created

save_file_to_drive('logseer.keras', MODEL_SAVE_PATH)
save_file_to_drive('tokenizer.pickle', TOKENIZER_PATH)